# 08 — Recommendation system

Combines:
- Annual forecast of the model selected per series (notebook 06).
- Actual usage of the last observed year (notebook 06).
- Cluster label (notebook 07).
- Disciplinary-relevance score (built here from the covariates).
- Publisher's SJR quartile (built here from the static covariates).
- Confidence metric (historical MASE).

Produces one recommendation per series in three categories: `validated_investment`, `renegotiation`, `promotion_deepening`.

See the full logic in [docs/06_clustering_recommendations.md](../docs/06_clustering_recommendations.md).

## 1. Configuration

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

OUTPUTS = REPO_ROOT / 'outputs'
OUTPUT_DIR = OUTPUTS / 'recommendations'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load inputs from previous notebooks

`series_profiles.csv` comes from notebook 07; `best_model_per_series.csv`,
`next_predictions.csv` and `actual_usage_last_year.csv` come from notebook 06.

In [ ]:
profiles = pd.read_csv(OUTPUTS / 'clustering' / 'series_profiles.csv')
best_model = pd.read_csv(OUTPUTS / 'comparison' / 'best_model_per_series.csv')
next_predictions = pd.read_csv(OUTPUTS / 'comparison' / 'next_predictions.csv')
actual_usage_last = pd.read_csv(OUTPUTS / 'comparison' / 'actual_usage_last_year.csv')

## 3. Build relevance score and dominant SJR from the covariates

- **Relevance score** per `(institution, publisher)`: dot product between the
  institution's enrollment shares (`prop_area_*`, latest available year) and the
  publisher's catalog shares (`kbart_area_*`). Measures disciplinary fit.
- **Dominant SJR** per publisher: taken from the `dominant_sjr` column of the
  static covariates.

If you do not have covariate files, replace this cell with neutral defaults
(`relevance_score = NaN`, `dominant_sjr = NaN`); the decision rule degrades gracefully.

In [ ]:
cov_dyn = pd.read_parquet(REPO_ROOT / 'data' / 'dynamic_covariates.parquet')
cov_stat = pd.read_parquet(REPO_ROOT / 'data' / 'static_covariates.parquet')

cols_dyn = [c for c in cov_dyn.columns if c.startswith('prop_area_')]
cols_stat = [c for c in cov_stat.columns if c.startswith('kbart_area_')]

# Latest enrollment profile per institution
latest_year = cov_dyn.groupby('institution')['year'].transform('max')
enrollment = (
    cov_dyn[cov_dyn['year'] == latest_year]
    .set_index('institution')[cols_dyn]
)
catalog = cov_stat.set_index('publisher')[cols_stat]
catalog.columns = cols_dyn  # align areas positionally

rel_rows = []
for sid in profiles['series_id']:
    inst, pub = sid.split('__')
    if inst in enrollment.index and pub in catalog.index:
        score = float(np.dot(enrollment.loc[inst].values, catalog.loc[pub].values))
    else:
        score = np.nan
    rel_rows.append({'series_id': sid, 'relevance_score': score})
relevance = pd.DataFrame(rel_rows)
relevance.to_csv(OUTPUTS / 'comparison' / 'relevance_score.csv', index=False)

sjr = cov_stat[['publisher', 'dominant_sjr']].drop_duplicates()
sjr.to_csv(OUTPUTS / 'comparison' / 'dominant_sjr_by_publisher.csv', index=False)

## 4. Per-series forecast-model selection

Rule: if adding covariates to TimesFM reduces MASE by at least 1%, use the covariate version; otherwise use the best historical model.

In [ ]:
impact_cov = pd.read_csv(OUTPUTS / 'timesfm_cov' / 'impact_covariates.csv')

def select_model(row):
    if row.get('improvement_pct', 0) <= -1.0:
        return 'TimesFM_cov'
    return row['best_model']

selection = best_model.merge(impact_cov[['series_id', 'improvement_pct']], on='series_id', how='left')
selection['selected_model'] = selection.apply(select_model, axis=1)
selection.to_csv(OUTPUT_DIR / 'model_selection_rule.csv', index=False)

## 5. Consolidate the decision table

In [ ]:
table = (
    selection[['series_id', 'selected_model', 'best_MASE']]
    .merge(profiles[['series_id', 'cluster_label']], on='series_id')
    .merge(actual_usage_last[['series_id', 'actual_usage']], on='series_id')
    .merge(next_predictions[['series_id', 'model', 'predicted_usage']]
             .rename(columns={'model': 'selected_model'}),
           on=['series_id', 'selected_model'])
    .merge(relevance[['series_id', 'relevance_score']], on='series_id', how='left')
)
table['publisher'] = table['series_id'].str.split('__').str[1]
table = table.merge(sjr, on='publisher', how='left')
table['change_pct'] = (table['predicted_usage'] - table['actual_usage']) / table['actual_usage'] * 100
table.head()

## 6. Confidence

In [ ]:
def compute_confidence(row):
    if pd.isna(row['best_MASE']):
        return 'no_data'
    if abs(row['change_pct']) > 80 or row['predicted_usage'] > 2 * row['actual_usage']:
        return 'no_data'
    if row['best_MASE'] > 1.5:
        return 'low'
    if row['best_MASE'] > 1.0:
        return 'medium'
    return 'high'

table['confidence'] = table.apply(compute_confidence, axis=1)

## 7. Decision rule

In [ ]:
def recommend(row):
    if row['confidence'] in ('low', 'no_data'):
        return 'promotion_deepening'

    if row['cluster_label'] == 'A_consolidated':
        threshold_inv, threshold_reneg = -10, -30
    else:
        threshold_inv, threshold_reneg = -5, -20

    if row.get('dominant_sjr') == 'Q1':
        threshold_reneg -= 10
    elif row.get('dominant_sjr') == 'Q4':
        threshold_reneg += 10

    if pd.notna(row.get('relevance_score')) and row['relevance_score'] < 0.10:
        threshold_reneg += 5

    if row['change_pct'] >= threshold_inv:
        return 'validated_investment'
    if row['change_pct'] < threshold_reneg:
        return 'renegotiation'
    return 'promotion_deepening'

table['recommendation'] = table.apply(recommend, axis=1)
table.to_csv(OUTPUT_DIR / 'annual_recommendations.csv', index=False)
table['recommendation'].value_counts()

## 8. Summary by cluster

In [ ]:
summary = table.groupby(['cluster_label', 'recommendation']).size().unstack(fill_value=0)
summary['total'] = summary.sum(axis=1)
summary.to_csv(OUTPUT_DIR / 'recommendations_summary_by_cluster.csv')
summary